# 09 — Raw particle-data inventory

## Purpose

This notebook constructs a reproducible metadata inventory of the raw
bidisperse rotating-drum dataset used for the particle-first ROM route.

At this stage, the large simulation files remain cloud-only. The notebook
inspects filenames, directory structure and logical file sizes without reading
multi-gigabyte file contents.

## Scientific objectives

1. Identify the available training and reserved-test simulations.
2. Catalogue the associated file types for each parameter value.
3. verify parameter parsing and train/test separation.
4. Detect missing, duplicated or anomalous files.
5. Select representative files for controlled format inspection.

No particle snapshots or ROM results are produced in this notebook stage.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

CLOUD_ROOT = (
    Path.home()
    / "Library"
    / "CloudStorage"
    / "OneDrive-TheUniversityofManchester"
    / "Anthony Thornton's files - data"
    / "particle_simulation_data"
)

TRAIN_DIR = CLOUD_ROOT / "bidisperse_mixtures"
TEST_DIR = CLOUD_ROOT / "bidisperse_mixtures_test_set"

print("Project root:", PROJECT_ROOT)
print("Cloud root:", CLOUD_ROOT)
print("Cloud root exists:", CLOUD_ROOT.exists())
print("Training directory exists:", TRAIN_DIR.exists())
print("Test directory exists:", TEST_DIR.exists())

Project root: /Users/rallen/Documents/msc-rom-particle-systems
Cloud root: /Users/rallen/Library/CloudStorage/OneDrive-TheUniversityofManchester/Anthony Thornton's files - data/particle_simulation_data
Cloud root exists: True
Training directory exists: True
Test directory exists: True


In [2]:
def classify_location(path: Path) -> tuple[str, str]:
    relative = path.relative_to(CLOUD_ROOT)
    top_folder = relative.parts[0]

    if top_folder == "bidisperse_mixtures_test_set":
        return "test", "raw_simulation_files"

    if "CG_windows_sigma" in relative.parts:
        return "training", "CG_windows_sigma"

    if "CG_windows" in relative.parts:
        return "training", "CG_windows"

    return "training", "raw_simulation_files"


records = []

for path in sorted(CLOUD_ROOT.rglob("*")):
    if not path.is_file():
        continue

    split, data_group = classify_location(path)

    records.append(
        {
            "split": split,
            "data_group": data_group,
            "filename": path.name,
            "extension": path.suffix.lower() or "<none>",
            "logical_size_bytes": path.stat().st_size,
            "relative_path": str(path.relative_to(CLOUD_ROOT)),
        }
    )

inventory = pd.DataFrame(records)
inventory["logical_size_MiB"] = (
    inventory["logical_size_bytes"] / 1024**2
)

print("Total inventoried files:", len(inventory))
display(inventory.head())

Total inventoried files: 897


,split,data_group,filename,extension,logical_size_bytes,relative_path,logical_size_MiB
0,training,raw_simulation_files,000002.s15.eps,.eps,2186101,bidisperse_mixtures/000002.s15.eps,2.084828
1,training,raw_simulation_files,000007.s18.eps,.eps,2186101,bidisperse_mixtures/000007.s18.eps,2.084828
2,training,raw_simulation_files,000008.s12.eps,.eps,2186101,bidisperse_mixtures/000008.s12.eps,2.084828
3,training,CG_windows,rotating_drum_bidisperse_size_ratio_1_000000.1...,.stat,1341706,bidisperse_mixtures/CG_windows/rotating_drum_b...,1.279551
4,training,CG_windows,rotating_drum_bidisperse_size_ratio_1_000000.1...,.stat,1340189,bidisperse_mixtures/CG_windows/rotating_drum_b...,1.278104


In [3]:
summary = (
    inventory
    .groupby(
        ["split", "data_group", "extension"],
        dropna=False
    )
    .agg(
        file_count=("filename", "size"),
        logical_size_GiB=("logical_size_bytes",
                          lambda x: x.sum() / 1024**3),
        minimum_size_MiB=("logical_size_MiB", "min"),
        maximum_size_MiB=("logical_size_MiB", "max"),
    )
    .reset_index()
    .sort_values(
        ["split", "data_group", "file_count"],
        ascending=[True, True, False],
    )
)

display(summary)

print(
    "\nTotal cloud-reported logical size:",
    f"{inventory['logical_size_bytes'].sum() / 1024**3:.2f} GiB"
)

,split,data_group,extension,file_count,logical_size_GiB,minimum_size_MiB,maximum_size_MiB
2,test,raw_simulation_files,.out,12,5.174670e-04,0.000554,0.054245
0,test,raw_simulation_files,.data,10,2.437173e+01,1991.458167,3379.161699
1,test,raw_simulation_files,.ene,10,1.270343e-03,0.130083,0.130083
3,test,raw_simulation_files,.restart,10,1.429873e-01,12.119931,19.734757
4,test,raw_simulation_files,.txt,10,6.426126e-08,0.000006,0.000007
5,test,raw_simulation_files,.xballs,10,2.536923e-06,0.000259,0.000260
6,test,raw_simulation_files,<none>,1,5.160627e-02,52.844825,52.844825
7,training,CG_windows,.stat,101,1.281829e-01,1.269444,1.344865
8,training,CG_windows_sigma,.stat,127,1.584241e-01,0.000231,1.341805
12,training,raw_simulation_files,.out,101,5.172803e-03,0.000525,0.055742



Total cloud-reported logical size: 282.56 GiB


In [4]:
simulation_pattern = re.compile(
    r"size_ratio_(\d+)[_.](\d+)_num_(\d+)"
)

def parse_simulation_filename(filename: str):
    match = simulation_pattern.search(filename)

    if match is None:
        return pd.Series(
            {
                "size_ratio": np.nan,
                "simulation_number": pd.NA,
                "parameter_parsed": False,
            }
        )

    integer_part, fractional_part, simulation_number = match.groups()

    return pd.Series(
        {
            "size_ratio": float(f"{integer_part}.{fractional_part}"),
            "simulation_number": int(simulation_number),
            "parameter_parsed": True,
        }
    )


parsed = inventory["filename"].apply(parse_simulation_filename)
inventory = pd.concat([inventory, parsed], axis=1)

print("Parsed files:", int(inventory["parameter_parsed"].sum()))
print("Unparsed files:", int((~inventory["parameter_parsed"]).sum()))

display(
    inventory.loc[
        ~inventory["parameter_parsed"],
        ["split", "data_group", "filename",
         "extension", "logical_size_MiB"],
    ].sort_values(["split", "data_group", "filename"])
)

Parsed files: 551
Unparsed files: 346


,split,data_group,filename,extension,logical_size_MiB
834,test,raw_simulation_files,StartSpheresBi_Con,<none>,52.844825
885,test,raw_simulation_files,slurm-34842.out,.out,0.000554
886,test,raw_simulation_files,slurm-34844.out,.out,0.052714
887,test,raw_simulation_files,slurm-34845.out,.out,0.052376
888,test,raw_simulation_files,slurm-34846.out,.out,0.054245
...,...,...,...,...,...
829,training,raw_simulation_files,slurm-27763.out,.out,0.054844
830,training,raw_simulation_files,slurm-27764.out,.out,0.054660
831,training,raw_simulation_files,slurm-27765.out,.out,0.055132
832,training,raw_simulation_files,slurm-27766.out,.out,0.000525


In [5]:
core_extensions = [".data", ".ene", ".restart", ".txt", ".xballs"]

parsed_raw = inventory.loc[
    inventory["parameter_parsed"]
    & inventory["data_group"].eq("raw_simulation_files")
].copy()

bundle_counts = (
    parsed_raw
    .groupby(
        ["split", "size_ratio", "simulation_number", "extension"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for extension in core_extensions:
    if extension not in bundle_counts.columns:
        bundle_counts[extension] = 0

bundle_counts["core_bundle_complete"] = (
    bundle_counts[core_extensions].eq(1).all(axis=1)
)

print("Parsed simulation bundles by split:")
display(
    bundle_counts.groupby("split").agg(
        simulations=("simulation_number", "size"),
        complete_core_bundles=("core_bundle_complete", "sum"),
        minimum_size_ratio=("size_ratio", "min"),
        maximum_size_ratio=("size_ratio", "max"),
    )
)

print("\nIncomplete or duplicated core bundles:")
display(
    bundle_counts.loc[
        ~bundle_counts["core_bundle_complete"],
        ["split", "size_ratio", "simulation_number"]
        + core_extensions,
    ]
)

Parsed simulation bundles by split:


,simulations,complete_core_bundles,minimum_size_ratio,maximum_size_ratio
split,,,,
test,10,10,1.0625,1.875
training,100,100,1.0000,2.000



Incomplete or duplicated core bundles:


extension,split,size_ratio,simulation_number,.data,.ene,.restart,.txt,.xballs


In [6]:
parameter_table = (
    bundle_counts[
        ["split", "size_ratio", "simulation_number"]
    ]
    .drop_duplicates()
    .sort_values(["split", "size_ratio"])
    .reset_index(drop=True)
)

train_parameters = parameter_table.loc[
    parameter_table["split"].eq("training"), "size_ratio"
].to_numpy()

test_parameters = parameter_table.loc[
    parameter_table["split"].eq("test"), "size_ratio"
].to_numpy()

overlap = np.intersect1d(train_parameters, test_parameters)

print("Training parameter count:", len(train_parameters))
print("Test parameter count:", len(test_parameters))
print("Exact train/test parameter overlap:", overlap.tolist())

print("\nTraining size-ratio range:")
print(train_parameters.min(), "to", train_parameters.max())

print("\nTest size ratios:")
print(test_parameters)

print("\nTraining grid-spacing summary:")
display(pd.Series(np.diff(train_parameters)).describe())

Training parameter count: 100
Test parameter count: 10
Exact train/test parameter overlap: []

Training size-ratio range:
1.0 to 2.0

Test size ratios:
[1.0625 1.125  1.25   1.3125 1.375  1.5    1.5625 1.625  1.75   1.875 ]

Training grid-spacing summary:


count    9.900000e+01
mean     1.010101e-02
std      1.005038e-07
min      1.010100e-02
25%      1.010100e-02
50%      1.010100e-02
75%      1.010100e-02
max      1.010200e-02
dtype: float64

In [7]:
cg_inventory = inventory.loc[
    inventory["data_group"].isin(
        ["CG_windows", "CG_windows_sigma"]
    )
].copy()

for group_name, group in cg_inventory.groupby(
    "data_group", sort=True
):
    names = sorted(group["filename"].tolist())

    print(f"\n{group_name}: {len(names)} files")
    print("\nFirst five filenames:")
    for name in names[:5]:
        print("  ", name)

    print("\nLast five filenames:")
    for name in names[-5:]:
        print("  ", name)


CG_windows: 101 files

First five filenames:
   rotating_drum_bidisperse_size_ratio_1_000000.1_w1.stat
   rotating_drum_bidisperse_size_ratio_1_000000.1_w10.stat
   rotating_drum_bidisperse_size_ratio_1_000000.1_w2.stat
   rotating_drum_bidisperse_size_ratio_1_000000.1_w3.stat
   rotating_drum_bidisperse_size_ratio_1_000000.1_w4.stat

Last five filenames:
   rotating_drum_bidisperse_size_ratio_1_909091.91_w5.stat
   rotating_drum_bidisperse_size_ratio_1_909091.91_w6.stat
   rotating_drum_bidisperse_size_ratio_1_909091.91_w7.stat
   rotating_drum_bidisperse_size_ratio_1_909091.91_w8.stat
   rotating_drum_bidisperse_size_ratio_1_909091.91_w9.stat

CG_windows_sigma: 127 files

First five filenames:
   rotating_drum_bidisperse_size_ratio_1_303030.31_t1032.43_1034.43.stat
   rotating_drum_bidisperse_size_ratio_1_303030.31_t1032.43_1043.19.stat
   rotating_drum_bidisperse_size_ratio_1_303030.31_t1032.43_1053.94.stat
   rotating_drum_bidisperse_size_ratio_1_303030.31_t1032.43_1075.45.stat
  

In [8]:
unparsed = inventory.loc[
    ~inventory["parameter_parsed"]
].copy()

def classify_unparsed(row):
    filename = row["filename"].lower()

    if row["data_group"] == "CG_windows":
        return "CG_windows field"

    if row["data_group"] == "CG_windows_sigma":
        return "CG_windows_sigma field"

    if filename.startswith("slurm-") and row["extension"] == ".out":
        return "Slurm output"

    if row["extension"] == ".eps":
        return "EPS figure"

    if filename == "xballs.txt":
        return "xballs configuration"

    if row["extension"] == "<none>":
        return "extensionless executable/configuration"

    return "other"


unparsed["unparsed_category"] = unparsed.apply(
    classify_unparsed, axis=1
)

display(
    unparsed.groupby(
        ["split", "unparsed_category"]
    )
    .agg(
        file_count=("filename", "size"),
        logical_size_MiB=("logical_size_MiB", "sum"),
    )
    .reset_index()
    .sort_values(["split", "file_count"], ascending=[True, False])
)

print("\nNon-Slurm, non-CG unparsed files:")
display(
    unparsed.loc[
        ~unparsed["unparsed_category"].isin(
            [
                "Slurm output",
                "CG_windows field",
                "CG_windows_sigma field",
            ]
        ),
        [
            "split",
            "filename",
            "extension",
            "logical_size_MiB",
            "unparsed_category",
        ],
    ]
)

,split,unparsed_category,file_count,logical_size_MiB
0,test,Slurm output,12,0.529886
1,test,extensionless executable/configuration,1,52.844825
3,training,CG_windows_sigma field,127,162.226295
2,training,CG_windows field,101,131.259244
5,training,Slurm output,101,5.296950
4,training,EPS figure,3,6.254485
6,training,xballs configuration,1,0.013335



Non-Slurm, non-CG unparsed files:


,split,filename,extension,logical_size_MiB,unparsed_category
0,training,000002.s15.eps,.eps,2.084828,EPS figure
1,training,000007.s18.eps,.eps,2.084828,EPS figure
2,training,000008.s12.eps,.eps,2.084828,EPS figure
833,training,xballs.txt,.txt,0.013335,xballs configuration
834,test,StartSpheresBi_Con,<none>,52.844825,extensionless executable/configuration


In [9]:
cg_window_pattern = re.compile(
    r"size_ratio_(\d+)[_.](\d+)\.(\d+)_w(\d+)\.stat$"
)

cg_sigma_pattern = re.compile(
    r"size_ratio_(\d+)[_.](\d+)\.(\d+)"
    r"_t(\d+(?:\.\d+)?)_(\d+(?:\.\d+)?)\.stat$"
)


def parse_cg_filename(filename):
    standard_match = cg_window_pattern.search(filename)

    if standard_match:
        integer, fraction, simulation, window = (
            standard_match.groups()
        )

        return {
            "cg_experiment": "numbered_windows",
            "size_ratio": float(f"{integer}.{fraction}"),
            "source_simulation": int(simulation),
            "window_number": int(window),
            "start_time": np.nan,
            "end_time": np.nan,
        }

    sigma_match = cg_sigma_pattern.search(filename)

    if sigma_match:
        integer, fraction, simulation, start, end = (
            sigma_match.groups()
        )

        return {
            "cg_experiment": "time_interval_windows",
            "size_ratio": float(f"{integer}.{fraction}"),
            "source_simulation": int(simulation),
            "window_number": pd.NA,
            "start_time": float(start),
            "end_time": float(end),
        }

    return {
        "cg_experiment": "unparsed",
        "size_ratio": np.nan,
        "source_simulation": pd.NA,
        "window_number": pd.NA,
        "start_time": np.nan,
        "end_time": np.nan,
    }


cg_parsed = pd.DataFrame(
    [
        {
            **row.to_dict(),
            **parse_cg_filename(row["filename"]),
        }
        for _, row in cg_inventory.iterrows()
    ]
)

print(cg_parsed["cg_experiment"].value_counts())
print("\nUnparsed CG filenames:")
display(
    cg_parsed.loc[
        cg_parsed["cg_experiment"].eq("unparsed"),
        ["data_group", "filename"],
    ]
)

cg_experiment
time_interval_windows    127
numbered_windows         100
unparsed                   1
Name: count, dtype: int64

Unparsed CG filenames:


,data_group,filename
10,CG_windows,rotating_drum_bidisperse_size_ratio_1_000000_w...


In [10]:
numbered_windows = cg_parsed.loc[
    cg_parsed["cg_experiment"].eq("numbered_windows")
].copy()

numbered_summary = (
    numbered_windows
    .groupby(["size_ratio", "source_simulation"])
    .agg(
        file_count=("filename", "size"),
        distinct_windows=("window_number", "nunique"),
        first_window=("window_number", "min"),
        last_window=("window_number", "max"),
        logical_size_MiB=("logical_size_MiB", "sum"),
    )
    .reset_index()
    .sort_values(["size_ratio", "source_simulation"])
)

display(numbered_summary)

duplicate_numbered_keys = numbered_windows.duplicated(
    ["size_ratio", "source_simulation", "window_number"],
    keep=False,
)

print("Number of parameter/simulation groups:",
      len(numbered_summary))
print("Duplicate numbered-window keys:",
      int(duplicate_numbered_keys.sum()))

print("\nWindow numbers in each group:")
for keys, group in numbered_windows.groupby(
    ["size_ratio", "source_simulation"]
):
    windows = sorted(group["window_number"].astype(int).tolist())
    print(keys, ":", windows)

,size_ratio,source_simulation,file_count,distinct_windows,first_window,last_window,logical_size_MiB
0,1.000000,1,10,10,1,10,12.785887
1,1.101010,11,10,10,1,10,12.899963
2,1.202020,21,10,10,1,10,12.923110
3,1.303030,31,10,10,1,10,12.928414
4,1.404040,41,10,10,1,10,12.996327
5,1.505051,51,10,10,1,10,12.962990
6,1.606061,61,10,10,1,10,13.076715
7,1.707071,71,10,10,1,10,13.111266
8,1.808081,81,10,10,1,10,13.155280
9,1.909091,91,10,10,1,10,13.139740


Number of parameter/simulation groups: 10
Duplicate numbered-window keys: 0

Window numbers in each group:
(1.0, 1) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
(1.10101, 11) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
(1.20202, 21) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
(1.30303, 31) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
(1.40404, 41) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
(1.505051, 51) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
(1.606061, 61) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
(1.707071, 71) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
(1.808081, 81) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
(1.909091, 91) : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [11]:
time_windows = cg_parsed.loc[
    cg_parsed["cg_experiment"].eq("time_interval_windows")
].copy()

time_windows["duration"] = (
    time_windows["end_time"] - time_windows["start_time"]
)

print("Files:", len(time_windows))
print("Size ratios:", sorted(time_windows["size_ratio"].unique()))
print(
    "Source simulations:",
    sorted(time_windows["source_simulation"].unique())
)
print(
    "All intervals have positive duration:",
    bool((time_windows["duration"] > 0).all())
)

duplicate_time_keys = time_windows.duplicated(
    [
        "size_ratio",
        "source_simulation",
        "start_time",
        "end_time",
    ],
    keep=False,
)

print("Duplicate interval keys:",
      int(duplicate_time_keys.sum()))

print("\nTime-coordinate summary:")
display(
    time_windows[
        ["start_time", "end_time", "duration"]
    ].describe()
)

print("\nMost frequent interval durations:")
display(
    time_windows["duration"]
    .round(2)
    .value_counts()
    .sort_index()
    .rename_axis("duration")
    .reset_index(name="file_count")
)

Files: 127
Size ratios: [1.30303]
Source simulations: [31]
All intervals have positive duration: True
Duplicate interval keys: 0

Time-coordinate summary:


,start_time,end_time,duration
count,127.000000,127.000000,127.000000
mean,1357.608189,1390.141575,32.533386
std,201.674771,202.086373,51.325711
min,1032.430000,1034.430000,2.000000
25%,1188.375000,1209.880000,10.750000
50%,1365.820000,1376.580000,10.760000
75%,1532.515000,1554.025000,21.510000
max,1709.970000,1720.720000,344.150000



Most frequent interval durations:


,duration,file_count
0,2.00,1
1,10.75,35
2,10.76,29
3,21.50,3
4,21.51,29
5,43.01,3
6,43.02,13
7,86.03,3
8,86.04,5
9,172.07,3


In [12]:
particle_data_files = inventory.loc[
    inventory["extension"].eq(".data")
    & inventory["parameter_parsed"]
].copy()

particle_data_files["logical_size_GiB"] = (
    particle_data_files["logical_size_bytes"] / 1024**3
)

display(
    particle_data_files.groupby("split").agg(
        simulations=("filename", "size"),
        total_size_GiB=("logical_size_GiB", "sum"),
        minimum_file_size_GiB=("logical_size_GiB", "min"),
        mean_file_size_GiB=("logical_size_GiB", "mean"),
        maximum_file_size_GiB=("logical_size_GiB", "max"),
    )
)

for split, group in particle_data_files.groupby("split"):
    correlation = group[
        ["size_ratio", "logical_size_GiB"]
    ].corr().iloc[0, 1]

    print(
        f"{split} correlation between size ratio "
        f"and .data size: {correlation:.6f}"
    )

print("\nSmallest candidate files:")
display(
    particle_data_files.nsmallest(
        5, "logical_size_bytes"
    )[
        [
            "split",
            "size_ratio",
            "simulation_number",
            "filename",
            "logical_size_GiB",
        ]
    ]
)

,simulations,total_size_GiB,minimum_file_size_GiB,mean_file_size_GiB,maximum_file_size_GiB
split,,,,,
test,10,24.371729,1.944783,2.437173,3.299963
training,100,256.187979,1.933997,2.561880,3.662722


test correlation between size ratio and .data size: 0.979644
training correlation between size ratio and .data size: 0.979256

Smallest candidate files:


,split,size_ratio,simulation_number,filename,logical_size_GiB
231,training,1.000000,1,rotating_drum_bidisperse_size_ratio_1_000000_n...,1.933997
237,training,1.010101,2,rotating_drum_bidisperse_size_ratio_1_010101_n...,1.934222
242,training,1.020202,3,rotating_drum_bidisperse_size_ratio_1_020202_n...,1.935109
247,training,1.030303,4,rotating_drum_bidisperse_size_ratio_1_030303_n...,1.936594
252,training,1.040404,5,rotating_drum_bidisperse_size_ratio_1_040404_n...,1.938578


In [13]:
unparsed_cg = cg_parsed.loc[
    cg_parsed["cg_experiment"].eq("unparsed")
]

for _, row in unparsed_cg.iterrows():
    print("Filename:", row["filename"])
    print("Logical size:", f"{row['logical_size_MiB']:.6f} MiB")
    print("Relative path:", row["relative_path"])

Filename: rotating_drum_bidisperse_size_ratio_1_000000_w1.stat
Logical size: 1.279551 MiB
Relative path: bidisperse_mixtures/CG_windows/rotating_drum_bidisperse_size_ratio_1_000000_w1.stat


In [14]:
import shutil

pilot_ratio = 1.0
pilot_simulation = 1

pilot_source_files = inventory.loc[
    inventory["split"].eq("training")
    & inventory["data_group"].eq("raw_simulation_files")
    & inventory["parameter_parsed"]
    & inventory["size_ratio"].eq(pilot_ratio)
    & inventory["simulation_number"].eq(pilot_simulation)
    & ~inventory["extension"].eq(".data")
].copy()

pilot_destination = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "particle_dem"
    / "pilot_size_ratio_1_000000_num_1"
)

pilot_destination.mkdir(parents=True, exist_ok=True)

print("Files selected:")
display(
    pilot_source_files[
        ["filename", "extension", "logical_size_MiB"]
    ]
)

print(
    "Total download/copy size:",
    f"{pilot_source_files['logical_size_MiB'].sum():.2f} MiB"
)

for relative_path in pilot_source_files["relative_path"]:
    source = CLOUD_ROOT / relative_path
    destination = pilot_destination / source.name

    shutil.copy2(source, destination)
    print("Copied:", destination.name)

print("\nPilot directory:", pilot_destination)

Files selected:


,filename,extension,logical_size_MiB
232,rotating_drum_bidisperse_size_ratio_1_000000_n...,.ene,0.130083
233,rotating_drum_bidisperse_size_ratio_1_000000_n...,.restart,12.018240
234,rotating_drum_bidisperse_size_ratio_1_000000_n...,.stat,0.000213
235,rotating_drum_bidisperse_size_ratio_1_000000_n...,.txt,0.000007
236,rotating_drum_bidisperse_size_ratio_1_000000_n...,.xballs,0.000294


Total download/copy size: 12.15 MiB
Copied: rotating_drum_bidisperse_size_ratio_1_000000_num_1.ene
Copied: rotating_drum_bidisperse_size_ratio_1_000000_num_1.restart
Copied: rotating_drum_bidisperse_size_ratio_1_000000_num_1.stat
Copied: rotating_drum_bidisperse_size_ratio_1_000000_num_1.txt
Copied: rotating_drum_bidisperse_size_ratio_1_000000_num_1.xballs

Pilot directory: /Users/rallen/Documents/msc-rom-particle-systems/data/raw/particle_dem/pilot_size_ratio_1_000000_num_1


In [15]:
from itertools import islice

def preview_text_file(path: Path, number_of_lines=8):
    print("\n" + "=" * 80)
    print(path.name)
    print("Size:", f"{path.stat().st_size / 1024**2:.3f} MiB")
    print("=" * 80)

    try:
        with path.open(
            "r",
            encoding="utf-8",
            errors="replace",
        ) as handle:
            for line_number, line in enumerate(
                islice(handle, number_of_lines),
                start=1,
            ):
                shortened = line.rstrip("\n")
                if len(shortened) > 300:
                    shortened = shortened[:300] + " ... [truncated]"

                print(f"{line_number:02d}: {shortened}")

    except Exception as error:
        print("Preview failed:", repr(error))


for path in sorted(pilot_destination.iterdir()):
    if path.is_file():
        preview_text_file(path)


rotating_drum_bidisperse_size_ratio_1_000000_num_1.ene
Size: 0.130 MiB
01:            time    gravitEnergy traKineticEnergy rotKineticEnergy   elasticEnergy   centerOfMassX   centerOfMassY   centerOfMassZ
02:                0     -179278.4779    0.03367211156     0.3508296029      38.16904318    0.04999387946   0.002587648818     -16.59663161
03:      1.720516059     -178908.3887      3620.329836      1.164727265      37.22622372    -0.7989306178   0.002915549912     -16.56237076
04:      3.441032118     -178019.8541      3610.246105      1.029792516      36.73201863     -1.837235446   0.002923790517     -16.48011503
05:      5.161548178     -176422.9118      3610.151127      1.025344018       36.6737156     -2.868232688   0.002914471775     -16.33227875
06:      6.882064237     -174129.7787      3610.217854      1.025006937      36.59406306     -3.887903871   0.002917221809      -16.1199929
07:      8.602580296     -171150.0808      3609.664762      1.037098004      36.53789269     -

In [16]:
tables_directory = PROJECT_ROOT / "results" / "tables"
tables_directory.mkdir(parents=True, exist_ok=True)

raw_inventory_path = (
    tables_directory / "raw_particle_file_inventory.csv"
)

cg_inventory_path = (
    tables_directory / "cg_window_file_inventory.csv"
)

inventory.sort_values(
    ["split", "data_group", "filename"]
).to_csv(
    raw_inventory_path,
    index=False,
)

cg_parsed.sort_values(
    [
        "cg_experiment",
        "size_ratio",
        "source_simulation",
        "start_time",
        "window_number",
    ],
    na_position="last",
).to_csv(
    cg_inventory_path,
    index=False,
)

print("Saved:", raw_inventory_path)
print("Rows:", len(inventory))

print("\nSaved:", cg_inventory_path)
print("Rows:", len(cg_parsed))

Saved: /Users/rallen/Documents/msc-rom-particle-systems/results/tables/raw_particle_file_inventory.csv
Rows: 897

Saved: /Users/rallen/Documents/msc-rom-particle-systems/results/tables/cg_window_file_inventory.csv
Rows: 228


In [17]:
restart_path = next(
    pilot_destination.glob("*.restart")
)

restart_line_records = []

with restart_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
) as handle:
    for line_number, line in enumerate(handle, start=1):
        stripped = line.strip()
        first_token = stripped.split(maxsplit=1)[0] if stripped else "<blank>"

        restart_line_records.append(
            {
                "line_number": line_number,
                "first_token": first_token,
                "character_count": len(line.rstrip("\n")),
                "preview": (
                    stripped[:250]
                    + (" ..." if len(stripped) > 250 else "")
                ),
            }
        )

restart_structure = pd.DataFrame(restart_line_records)

print("Restart file:", restart_path.name)
print("Total lines:", len(restart_structure))
print(
    "Maximum line length:",
    restart_structure["character_count"].max()
)

print("\nMost frequent first tokens:")
display(
    restart_structure["first_token"]
    .value_counts()
    .head(20)
    .rename_axis("first_token")
    .reset_index(name="line_count")
)

Restart file: rotating_drum_bidisperse_size_ratio_1_000000_num_1.restart
Total lines: 24035
Maximum line length: 581

Most frequent first tokens:


,first_token,line_count
0,LinearViscoelasticFrictionInteraction,13201
1,BaseParticle,10792
2,LinearViscoelasticFrictionMixedSpecies,15
3,LinearViscoelasticFrictionSpecies,6
4,InfiniteWall,2
5,CubeInsertionBoundary,2
6,MercuryDPM,1
7,dataFile,1
8,fStatFile,1
9,eneFile,1


In [18]:
common_record_tokens = set(
    restart_structure["first_token"]
    .value_counts()
    .head(5)
    .index
)

possible_headers = restart_structure.loc[
    ~restart_structure["first_token"].isin(common_record_tokens)
].copy()

print("Possible structural/header lines:")
for _, row in possible_headers.head(60).iterrows():
    print(
        f"{row['line_number']:>6}: "
        f"{row['preview']}"
    )

print("\nLongest lines:")
display(
    restart_structure.nlargest(
        10, "character_count"
    )[
        [
            "line_number",
            "first_token",
            "character_count",
            "preview",
        ]
    ]
)

Possible structural/header lines:
     1: MercuryDPM 1.0 runNumber 1 name RotatingDrumBidisperseParameterStudy1.000000.1 revision e22cf5eb repository https://bitbucket.org/mercurydpm/mercurydpm.git
     2: dataFile    fileType ONE_FILE saveCount 7454 counter 1002 lastSavedTimeStep 7454889
     3: fStatFile   fileType NO_FILE saveCount 7454 counter 0
     4: eneFile     fileType ONE_FILE saveCount 7454 counter 1002 lastSavedTimeStep 7454889
     5: restartFile fileType ONE_FILE saveCount 4294967295 counter 2 lastSavedTimeStep 7454889
     6: statFile    fileType ONE_FILE saveCount 7454 counter 0
     7: interactionFile fileType NO_FILE saveCount 0 counter 0
     8: xMin -30 xMax 30 yMin -5 yMax 5 zMin -30 zMax 30
     9: timeStep 0.000230817823879522 time 1720.72125622023 ntimeSteps 7454889 timeMax 1720.72116286364
    10: systemDimensions 3 particleDimensions 3 gravity 0 0 -1 backgroundDrag 0 writeVTK 0 NO_FILE NO_FILE 0 0 0 0 0 random  0 1103515245 12345 1073741824 607 273 176025410 n

,line_number,first_token,character_count,preview
13960,13961,LinearViscoelasticFrictionInteraction,581,LinearViscoelasticFrictionInteraction particle...
16074,16075,LinearViscoelasticFrictionInteraction,579,LinearViscoelasticFrictionInteraction particle...
16526,16527,LinearViscoelasticFrictionInteraction,579,LinearViscoelasticFrictionInteraction particle...
20891,20892,LinearViscoelasticFrictionInteraction,579,LinearViscoelasticFrictionInteraction particle...
13385,13386,LinearViscoelasticFrictionInteraction,578,LinearViscoelasticFrictionInteraction particle...
12283,12284,LinearViscoelasticFrictionInteraction,577,LinearViscoelasticFrictionInteraction particle...
17979,17980,LinearViscoelasticFrictionInteraction,577,LinearViscoelasticFrictionInteraction particle...
20648,20649,LinearViscoelasticFrictionInteraction,577,LinearViscoelasticFrictionInteraction particle...
14276,14277,LinearViscoelasticFrictionInteraction,576,LinearViscoelasticFrictionInteraction particle...
18850,18851,LinearViscoelasticFrictionInteraction,576,LinearViscoelasticFrictionInteraction particle...


In [19]:
ene_path = next(pilot_destination.glob("*.ene"))

energy = pd.read_csv(
    ene_path,
    sep=r"\s+",
)

print("Energy rows:", len(energy))
print("Columns:", energy.columns.tolist())
print("Duplicate times:", int(energy["time"].duplicated().sum()))
print("Times strictly increasing:",
      bool(energy["time"].is_monotonic_increasing))

print(
    "Time range:",
    float(energy["time"].min()),
    "to",
    float(energy["time"].max()),
)

time_steps = np.diff(energy["time"].to_numpy())

print("\nSaved-time spacing:")
display(pd.Series(time_steps).describe())

print("\nFinal five energy records:")
display(energy.tail())

Energy rows: 1002
Columns: ['time', 'gravitEnergy', 'traKineticEnergy', 'rotKineticEnergy', 'elasticEnergy', 'centerOfMassX', 'centerOfMassY', 'centerOfMassZ']
Duplicate times: 0
Times strictly increasing: True
Time range: 0.0 to 1720.721256

Saved-time spacing:


count    1001.000000
mean        1.719002
std         0.047895
min         0.205197
25%         1.720516
50%         1.720516
75%         1.720516
max         1.720517
dtype: float64


Final five energy records:


,time,gravitEnergy,traKineticEnergy,rotKineticEnergy,elasticEnergy,centerOfMassX,centerOfMassY,centerOfMassZ
997,1715.354511,-108586.7480,12374.05831,162.811725,36.077365,-12.352314,-0.004842,-10.052374
998,1717.075027,-108664.0449,11616.71851,154.883414,39.042181,-12.347801,-0.005127,-10.059529
999,1718.795543,-108831.8091,12345.16661,160.805024,36.789981,-12.332764,-0.007109,-10.075060
1000,1720.516059,-109186.6737,12410.56681,168.819973,39.569906,-12.319163,-0.004254,-10.107912
1001,1720.721256,-109218.8568,12413.00949,174.413764,38.889622,-12.318175,-0.003177,-10.110891


In [20]:
import hashlib

cg_folder = TRAIN_DIR / "CG_windows"

canonical_cg_path = (
    cg_folder
    / "rotating_drum_bidisperse_size_ratio_1_000000.1_w1.stat"
)

anomalous_cg_path = (
    cg_folder
    / "rotating_drum_bidisperse_size_ratio_1_000000_w1.stat"
)

def sha256_file(path: Path, chunk_size=1024**2):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


comparison = pd.DataFrame(
    {
        "file": [
            canonical_cg_path.name,
            anomalous_cg_path.name,
        ],
        "logical_size_bytes": [
            canonical_cg_path.stat().st_size,
            anomalous_cg_path.stat().st_size,
        ],
        "sha256": [
            sha256_file(canonical_cg_path),
            sha256_file(anomalous_cg_path),
        ],
    }
)

display(comparison)

print(
    "Byte-for-byte identical:",
    comparison["sha256"].nunique() == 1
)


,file,logical_size_bytes,sha256
0,rotating_drum_bidisperse_size_ratio_1_000000.1...,1341706,1e50cb9f723adbda1db27a1455d76bcc6818e653c09346...
1,rotating_drum_bidisperse_size_ratio_1_000000_w...,1341706,1e50cb9f723adbda1db27a1455d76bcc6818e653c09346...


Byte-for-byte identical: True


In [21]:
with restart_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
) as handle:
    restart_lines = [line.rstrip("\n") for line in handle]

print("Species and mixed-species records:")
for line_number in range(11, 33):
    print(f"{line_number:>4}: {restart_lines[line_number - 1]}")

particle_header_index = next(
    index
    for index, line in enumerate(restart_lines)
    if line.startswith("Particles ")
)

interaction_header_index = next(
    index
    for index, line in enumerate(restart_lines)
    if line.startswith("Interactions ")
)

print("\nParticle header:")
print(restart_lines[particle_header_index])

print("\nFirst three particle records:")
for line in restart_lines[
    particle_header_index + 1:
    particle_header_index + 4
]:
    print(line)

print("\nFinal particle record:")
print(restart_lines[interaction_header_index - 1])

Species and mixed-species records:
  11: Species 6
  12: LinearViscoelasticFrictionSpecies id 0 density 1.90985931710274 stiffness 200000 dissipation 50 slidingStiffness 57142.8571428571 slidingDissipation 14.2857142857143 slidingFrictionCoefficient 0.5 slidingFrictionCoefficientStatic 0.5 rollingStiffness 80000 rollingDissipation 20 rollingFrictionCoefficient 0.5 rollingFrictionCoefficientStatic 0.5 torsionStiffness 0 torsionDissipation 0 torsionFrictionCoefficient 0 torsionFrictionCoefficientStatic 0
  13: LinearViscoelasticFrictionSpecies id 1 density 1.90985931710274 stiffness 200000 dissipation 50 slidingStiffness 57142.8571428571 slidingDissipation 14.2857142857143 slidingFrictionCoefficient 0.5 slidingFrictionCoefficientStatic 0.5 rollingStiffness 80000 rollingDissipation 20 rollingFrictionCoefficient 0.5 rollingFrictionCoefficientStatic 0.5 torsionStiffness 0 torsionDissipation 0 torsionFrictionCoefficient 0 torsionFrictionCoefficientStatic 0
  14: LinearViscoelasticFrictionMix

In [22]:
def extract_scalar(tokens, label, conversion=float):
    index = tokens.index(label)
    return conversion(tokens[index + 1])


def extract_vector(tokens, label, length=3):
    index = tokens.index(label)
    return [
        float(value)
        for value in tokens[index + 1:index + 1 + length]
    ]


particle_records = []
particle_parse_errors = []

particle_lines = restart_lines[
    particle_header_index + 1:
    interaction_header_index
]

for record_number, line in enumerate(
    particle_lines,
    start=1,
):
    tokens = line.split()

    try:
        position = extract_vector(tokens, "position")
        velocity = extract_vector(tokens, "velocity")
        angular_velocity = extract_vector(
            tokens, "angularVelocity"
        )

        particle_records.append(
            {
                "id": extract_scalar(tokens, "id", int),
                "species_index": extract_scalar(
                    tokens, "indSpecies", int
                ),
                "x": position[0],
                "y": position[1],
                "z": position[2],
                "vx": velocity[0],
                "vy": velocity[1],
                "vz": velocity[2],
                "omega_x": angular_velocity[0],
                "omega_y": angular_velocity[1],
                "omega_z": angular_velocity[2],
                "radius": extract_scalar(tokens, "radius"),
            }
        )

    except Exception as error:
        particle_parse_errors.append(
            {
                "record_number": record_number,
                "error": repr(error),
                "preview": line[:300],
            }
        )


particles_final = pd.DataFrame(particle_records)

print("Parsed particle records:", len(particles_final))
print("Parse errors:", len(particle_parse_errors))

if particle_parse_errors:
    display(pd.DataFrame(particle_parse_errors).head())
else:
    display(particles_final.head())

Parsed particle records: 10792
Parse errors: 0


,id,species_index,x,y,z,vx,vy,vz,omega_x,omega_y,omega_z,radius
0,0,0,-6.499203,4.508166,-15.968287,-0.541099,-0.010186,0.122349,0.025914,0.016975,0.007942,0.491018
1,1,0,-19.738927,2.880610,-8.707140,-0.314091,-0.000229,0.716247,0.001514,0.034193,-0.000327,0.489359
2,2,0,-21.971803,3.993216,-15.967509,-0.582374,0.000255,0.801844,0.000865,0.037360,0.000047,0.489922
3,3,0,-26.421358,0.454192,6.900698,0.253792,0.000457,0.937535,-0.015477,0.023693,0.003465,0.520119
4,4,0,-15.514193,-0.701660,3.996931,0.727345,0.028907,-0.251385,0.038425,0.344620,0.325591,0.505970


In [23]:
reported_particle_count = int(
    restart_lines[particle_header_index].split()[1]
)

print("Reported particle count:", reported_particle_count)
print("Parsed particle count:", len(particles_final))
print(
    "Count agrees:",
    len(particles_final) == reported_particle_count
)

print("\nParticle-ID checks:")
print("Unique IDs:", particles_final["id"].nunique())
print("Duplicate IDs:", int(particles_final["id"].duplicated().sum()))
print("Minimum ID:", particles_final["id"].min())
print("Maximum ID:", particles_final["id"].max())

print("\nSpecies distribution:")
display(
    particles_final.groupby("species_index").agg(
        particle_count=("id", "size"),
        minimum_radius=("radius", "min"),
        mean_radius=("radius", "mean"),
        maximum_radius=("radius", "max"),
    )
)

print("\nPosition ranges:")
display(
    particles_final[
        ["x", "y", "z"]
    ].agg(["min", "max"])
)

print("\nMissing or non-finite values:")
numeric_values = particles_final.select_dtypes(
    include=[np.number]
).to_numpy()

print("Missing values:",
      int(particles_final.isna().sum().sum()))
print("All numeric values finite:",
      bool(np.isfinite(numeric_values).all()))

Reported particle count: 10792
Parsed particle count: 10792
Count agrees: True

Particle-ID checks:
Unique IDs: 10792
Duplicate IDs: 0
Minimum ID: 0
Maximum ID: 10791

Species distribution:


,particle_count,minimum_radius,mean_radius,maximum_radius
species_index,,,,
0,5396,0.475004,0.499741,0.524986
1,5396,0.475004,0.499741,0.524986



Position ranges:


,x,y,z
min,-29.523435,-4.525097,-29.519125
max,19.637037,4.524972,17.586009



Missing or non-finite values:
Missing values: 0
All numeric values finite: True


In [24]:
comparison_restart_row = inventory.loc[
    inventory["split"].eq("training")
    & inventory["parameter_parsed"]
    & inventory["simulation_number"].eq(100)
    & inventory["extension"].eq(".restart")
].iloc[0]

comparison_source = (
    CLOUD_ROOT
    / comparison_restart_row["relative_path"]
)

comparison_destination_directory = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "particle_dem"
    / "pilot_size_ratio_2_000000_num_100"
)

comparison_destination_directory.mkdir(
    parents=True,
    exist_ok=True,
)

comparison_restart_path = (
    comparison_destination_directory
    / comparison_source.name
)

if not comparison_restart_path.exists():
    shutil.copy2(
        comparison_source,
        comparison_restart_path,
    )

print("Copied:", comparison_restart_path)
print(
    "Size:",
    f"{comparison_restart_path.stat().st_size / 1024**2:.2f} MiB"
)

comparison_counts = {}

with comparison_restart_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
) as handle:
    for line in handle:
        if line.startswith("timeStep "):
            comparison_counts["time_record"] = line.strip()

        elif line.startswith("Species "):
            comparison_counts["species"] = int(
                line.split()[1]
            )

        elif line.startswith("Particles "):
            comparison_counts["particles"] = int(
                line.split()[1]
            )

        elif line.startswith("Interactions "):
            comparison_counts["interactions"] = int(
                line.split()[1]
            )

print("\nSize-ratio-2 restart summary:")
for key, value in comparison_counts.items():
    print(f"{key}: {value}")

Copied: /Users/rallen/Documents/msc-rom-particle-systems/data/raw/particle_dem/pilot_size_ratio_2_000000_num_100/rotating_drum_bidisperse_size_ratio_2.000000_num_100.restart
Size: 21.46 MiB

Size-ratio-2 restart summary:
time_record: timeStep 0.000126750177858053 time 1720.72124938943 ntimeSteps 13575691 timeMax 1720.72116286364
species: 6
particles: 20449
interactions: 22310


In [29]:
def parse_restart_file(path: Path):
    metadata = {}
    particle_lines = []
    inside_particles = False

    with path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as handle:
        for line in handle:
            stripped = line.strip()

            if stripped.startswith("timeStep "):
                tokens = stripped.split()
                metadata["time_step"] = float(tokens[1])
                metadata["final_time"] = float(
                    tokens[tokens.index("time") + 1]
                )
                metadata["number_of_time_steps"] = int(
                    tokens[tokens.index("ntimeSteps") + 1]
                )

            elif stripped.startswith("Species "):
                metadata["reported_species"] = int(
                    stripped.split()[1]
                )

            elif stripped.startswith("Particles "):
                metadata["reported_particles"] = int(
                    stripped.split()[1]
                )
                inside_particles = True
                continue

            elif stripped.startswith("Interactions "):
                metadata["reported_interactions"] = int(
                    stripped.split()[1]
                )
                inside_particles = False

            elif inside_particles:
                particle_lines.append(stripped)

    particle_records = []
    errors = []

    for record_number, line in enumerate(
        particle_lines,
        start=1,
    ):
        tokens = line.split()

        try:
            position = extract_vector(tokens, "position")
            velocity = extract_vector(tokens, "velocity")
            angular_velocity = extract_vector(
                tokens, "angularVelocity"
            )

            particle_records.append(
                {
                    "id": extract_scalar(tokens, "id", int),
                    "species_index": extract_scalar(
                        tokens, "indSpecies", int
                    ),
                    "x": position[0],
                    "y": position[1],
                    "z": position[2],
                    "vx": velocity[0],
                    "vy": velocity[1],
                    "vz": velocity[2],
                    "omega_x": angular_velocity[0],
                    "omega_y": angular_velocity[1],
                    "omega_z": angular_velocity[2],
                    "radius": extract_scalar(tokens, "radius"),
                }
            )

        except Exception as error:
            errors.append(
                {
                    "record_number": record_number,
                    "error": repr(error),
                    "preview": line[:250],
                }
            )

    return metadata, pd.DataFrame(particle_records), errors

In [26]:
restart_cases = {
    "ratio_1": restart_path,
    "ratio_2": comparison_restart_path,
}

restart_results = {}

for case_name, path in restart_cases.items():
    metadata, particles, errors = parse_restart_file(path)

    restart_results[case_name] = {
        "metadata": metadata,
        "particles": particles,
        "errors": errors,
    }

    print("\n" + "=" * 70)
    print(case_name, "-", path.name)
    print("=" * 70)

    print("Reported particles:",
          metadata.get("reported_particles"))
    print("Parsed particles:", len(particles))
    print("Parse errors:", len(errors))
    print("Unique IDs:", particles["id"].nunique())
    print("Duplicate IDs:",
          int(particles["id"].duplicated().sum()))
    print(
        "All numeric values finite:",
        bool(
            np.isfinite(
                particles.select_dtypes(
                    include=[np.number]
                ).to_numpy()
            ).all()
        ),
    )

    if errors:
        display(pd.DataFrame(errors).head())


ratio_1 - rotating_drum_bidisperse_size_ratio_1_000000_num_1.restart
Reported particles: 10792
Parsed particles: 10792
Parse errors: 0
Unique IDs: 10792
Duplicate IDs: 0
All numeric values finite: True

ratio_2 - rotating_drum_bidisperse_size_ratio_2.000000_num_100.restart
Reported particles: 20449
Parsed particles: 20449
Parse errors: 0
Unique IDs: 20449
Duplicate IDs: 0
All numeric values finite: True


In [27]:
species_comparisons = []

for case_name, result in restart_results.items():
    particles = result["particles"]

    summary = (
        particles
        .groupby("species_index")
        .agg(
            particle_count=("id", "size"),
            minimum_radius=("radius", "min"),
            mean_radius=("radius", "mean"),
            maximum_radius=("radius", "max"),
            minimum_id=("id", "min"),
            maximum_id=("id", "max"),
        )
        .reset_index()
    )

    summary.insert(0, "case", case_name)
    species_comparisons.append(summary)

species_comparison = pd.concat(
    species_comparisons,
    ignore_index=True,
)

display(species_comparison)

print("\nTotal particle counts:")
display(
    species_comparison.groupby("case").agg(
        total_particles=("particle_count", "sum"),
        particle_species_present=("species_index", "nunique"),
    )
)

,case,species_index,particle_count,minimum_radius,mean_radius,maximum_radius,minimum_id,maximum_id
0,ratio_1,0,5396,0.475004,0.499741,0.524986,0,5395
1,ratio_1,1,5396,0.475004,0.499741,0.524986,5396,10791
2,ratio_2,0,2280,0.633379,0.666021,0.699935,0,2279
3,ratio_2,1,18169,0.316667,0.333403,0.349999,2280,20448



Total particle counts:


,total_particles,particle_species_present
case,,
ratio_1,10792,2
ratio_2,20449,2


In [28]:
endpoint_summary_records = []

for case_name, result in restart_results.items():
    metadata = result["metadata"]
    particles = result["particles"]

    endpoint_summary_records.append(
        {
            "case": case_name,
            "time_step": metadata["time_step"],
            "final_time": metadata["final_time"],
            "number_of_time_steps":
                metadata["number_of_time_steps"],
            "particle_count": len(particles),
            "minimum_particle_id": particles["id"].min(),
            "maximum_particle_id": particles["id"].max(),
            "unique_particle_ids": particles["id"].nunique(),
            "interaction_count":
                metadata["reported_interactions"],
        }
    )

endpoint_summary = pd.DataFrame(
    endpoint_summary_records
)

display(endpoint_summary)

print(
    "Particle-count increase:",
    endpoint_summary.loc[
        endpoint_summary["case"].eq("ratio_2"),
        "particle_count",
    ].iloc[0]
    -
    endpoint_summary.loc[
        endpoint_summary["case"].eq("ratio_1"),
        "particle_count",
    ].iloc[0]
)

,case,time_step,final_time,number_of_time_steps,particle_count,minimum_particle_id,maximum_particle_id,unique_particle_ids,interaction_count
0,ratio_1,0.000231,1720.721256,7454889,10792,0,10791,10792,13201
1,ratio_2,0.000127,1720.721249,13575691,20449,0,20448,20449,22310


Particle-count increase: 9657


In [30]:
pilot_data_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "particle_dem"
    / "pilot_size_ratio_1_000000_num_1"
    / "rotating_drum_bidisperse_size_ratio_1_000000_num_1.data"
)

print("File exists:", pilot_data_path.exists())
print("File:", pilot_data_path)
print(
    "Size:",
    f"{pilot_data_path.stat().st_size / 1024**3:.6f} GiB"
)

File exists: True
File: /Users/rallen/Documents/msc-rom-particle-systems/data/raw/particle_dem/pilot_size_ratio_1_000000_num_1/rotating_drum_bidisperse_size_ratio_1_000000_num_1.data
Size: 1.933997 GiB


In [31]:
preview_records = []

with pilot_data_path.open(
    "r",
    encoding="utf-8",
    errors="replace",
) as handle:
    for line_number, line in enumerate(
        islice(handle, 15),
        start=1,
    ):
        stripped = line.rstrip("\n")

        preview_records.append(
            {
                "line_number": line_number,
                "token_count": len(stripped.split()),
                "character_count": len(stripped),
                "preview": (
                    stripped[:500]
                    + (" ..." if len(stripped) > 500 else "")
                ),
            }
        )

display(pd.DataFrame(preview_records))

,line_number,token_count,character_count,preview
0,1,8,27,10792 0 -30 -5 -30 30 5 30
1,2,14,202,-8.30478454970132 -0.576347414743027 -18.50961...
2,3,14,199,-2.5739234345844 2.68969013653904 -28.18686403...
3,4,14,204,2.32410990603487 3.88803155431166 -24.67495122...
4,5,14,205,8.33110152576051 1.67278721770076 -23.47668405...
5,6,14,201,17.8730754035964 3.29061998447955 -23.46057256...
6,7,14,205,14.1843070096909 -2.85877618669664 -18.5747143...
7,8,14,202,-21.9630492768456 3.08446322362048 -16.8790184...
8,9,14,210,-21.5443880121192 0.0668177274757502 -15.21650...
9,10,14,205,-22.0188691039883 -1.84806121324434 -15.042425...


In [32]:
for record in preview_records:
    print(
        record["line_number"],
        "| tokens:", record["token_count"],
        "|",
        record["preview"],
    )

1 | tokens: 8 | 10792 0 -30 -5 -30 30 5 30 
2 | tokens: 14 | -8.30478454970132 -0.576347414743027 -18.5096121484051 7.649831072e-05 -0.0004009847759 7.585814221e-06 0.491018326 0.691640493 0.6095558335 1.877308974 0.000208385563 -1.993968473e-05 4.023650781e-08 0
3 | tokens: 14 | -2.5739234345844 2.68969013653904 -28.1868640367064 7.963815083e-05 -2.614474088e-05 5.536013193e-05 0.4893590995 2.777870025 1.440738366 -2.712029256 1.453662495e-05 1.002170087e-05 5.30219817e-06 0
4 | tokens: 14 | 2.32410990603487 3.88803155431166 -24.6749512298831 6.559711404e-05 -2.762020258e-05 6.319781443e-05 0.4899224487 0.9201442527 0.2681206295 0.5673777793 -2.499680474e-05 2.287270717e-05 -4.312302011e-06 0
5 | tokens: 14 | 8.33110152576051 1.67278721770076 -23.4766840536304 -0.0001415689769 -4.047284726e-05 2.866153547e-05 0.5201189387 0.8324807818 -1.218131436 -2.074208801 -8.470831515e-06 -1.069307049e-05 1.169183927e-05 0
6 | tokens: 14 | 17.8730754035964 3.29061998447955 -23.4605725648808 2.962

In [33]:
snapshot_records = []

with pilot_data_path.open("rb") as handle:
    snapshot_number = 0

    while True:
        byte_offset = handle.tell()
        header_line = handle.readline()

        if not header_line:
            break

        header_tokens = header_line.split()

        if len(header_tokens) != 8:
            raise ValueError(
                f"Expected 8 header values at byte "
                f"{byte_offset}, found {len(header_tokens)}"
            )

        particle_count = int(header_tokens[0])
        snapshot_time = float(header_tokens[1])
        bounds = [
            float(value)
            for value in header_tokens[2:]
        ]

        for particle_row in range(particle_count):
            line = handle.readline()

            if not line:
                raise EOFError(
                    f"Unexpected end of file in snapshot "
                    f"{snapshot_number}, particle row "
                    f"{particle_row}"
                )

        snapshot_records.append(
            {
                "snapshot_number": snapshot_number,
                "byte_offset": byte_offset,
                "particle_count": particle_count,
                "time": snapshot_time,
                "x_min": bounds[0],
                "y_min": bounds[1],
                "z_min": bounds[2],
                "x_max": bounds[3],
                "y_max": bounds[4],
                "z_max": bounds[5],
            }
        )

        snapshot_number += 1

        if snapshot_number % 100 == 0:
            print("Scanned snapshots:", snapshot_number)

snapshot_index = pd.DataFrame(snapshot_records)

print("\nTotal snapshots:", len(snapshot_index))
display(snapshot_index.head())
display(snapshot_index.tail())

Scanned snapshots: 100
Scanned snapshots: 200
Scanned snapshots: 300
Scanned snapshots: 400
Scanned snapshots: 500
Scanned snapshots: 600
Scanned snapshots: 700
Scanned snapshots: 800
Scanned snapshots: 900
Scanned snapshots: 1000

Total snapshots: 1002


,snapshot_number,byte_offset,particle_count,time,x_min,y_min,z_min,x_max,y_max,z_max
0,0,0,10792,0.000000,-30.0,-5.0,-30.0,30.0,5.0,30.0
1,1,2222885,10792,1.720516,-30.0,-5.0,-30.0,30.0,5.0,30.0
2,2,4353329,10792,3.441032,-30.0,-5.0,-30.0,30.0,5.0,30.0
3,3,6490113,10792,5.161548,-30.0,-5.0,-30.0,30.0,5.0,30.0
4,4,8627985,10792,6.882064,-30.0,-5.0,-30.0,30.0,5.0,30.0


,snapshot_number,byte_offset,particle_count,time,x_min,y_min,z_min,x_max,y_max,z_max
997,997,2066255148,10792,1715.354511,-30.0,-5.0,-30.0,30.0,5.0,30.0
998,998,2068327929,10792,1717.075027,-30.0,-5.0,-30.0,30.0,5.0,30.0
999,999,2070398808,10792,1718.795543,-30.0,-5.0,-30.0,30.0,5.0,30.0
1000,1000,2072470410,10792,1720.516059,-30.0,-5.0,-30.0,30.0,5.0,30.0
1001,1001,2074541674,10792,1720.721256,-30.0,-5.0,-30.0,30.0,5.0,30.0


In [34]:
print("Particle-count distribution:")
display(
    snapshot_index["particle_count"]
    .value_counts()
    .sort_index()
    .rename_axis("particle_count")
    .reset_index(name="snapshot_count")
)

print("\nDuplicate snapshot times:",
      int(snapshot_index["time"].duplicated().sum()))
print(
    "Times strictly increasing:",
    bool(snapshot_index["time"].is_monotonic_increasing)
)

print("\nTime summary:")
display(snapshot_index["time"].describe())

print("\nComparison with energy timestamps:")
print("Same number of records:",
      len(snapshot_index) == len(energy))

if len(snapshot_index) == len(energy):
    time_difference = (
        snapshot_index["time"].to_numpy()
        - energy["time"].to_numpy()
    )

    print(
        "Maximum absolute time difference:",
        float(np.max(np.abs(time_difference)))
    )
    print(
        "All timestamps agree:",
        bool(np.allclose(
            snapshot_index["time"],
            energy["time"],
            rtol=0,
            atol=1e-8,
        ))
    )

print("\nDistinct domain bounds:")
display(
    snapshot_index[
        ["x_min", "y_min", "z_min",
         "x_max", "y_max", "z_max"]
    ].drop_duplicates()
)

Particle-count distribution:


,particle_count,snapshot_count
0,10792,1002



Duplicate snapshot times: 0
Times strictly increasing: True

Time summary:


count    1002.000000
mean      861.116775
std       497.909203
min         0.000000
25%       430.559144
50%       861.118288
75%      1291.677431
max      1720.721256
Name: time, dtype: float64


Comparison with energy timestamps:
Same number of records: True
Maximum absolute time difference: 0.0
All timestamps agree: True

Distinct domain bounds:


,x_min,y_min,z_min,x_max,y_max,z_max
0,-30.0,-5.0,-30.0,30.0,5.0,30.0


In [35]:
def read_particle_snapshot(path: Path, byte_offset: int):
    with path.open("rb") as handle:
        handle.seek(int(byte_offset))

        header = handle.readline().split()
        particle_count = int(header[0])
        snapshot_time = float(header[1])

        values = np.empty(
            (particle_count, 14),
            dtype=np.float64,
        )

        for row_index in range(particle_count):
            row = np.fromstring(
                handle.readline(),
                sep=" ",
                dtype=np.float64,
            )

            if row.size != 14:
                raise ValueError(
                    f"Particle row {row_index} has "
                    f"{row.size} values, expected 14"
                )

            values[row_index] = row

    return snapshot_time, values


first_time, first_snapshot = read_particle_snapshot(
    pilot_data_path,
    snapshot_index.iloc[0]["byte_offset"],
)

final_time, final_snapshot = read_particle_snapshot(
    pilot_data_path,
    snapshot_index.iloc[-1]["byte_offset"],
)

print("First snapshot:", first_time, first_snapshot.shape)
print("Final snapshot:", final_time, final_snapshot.shape)
print("All first-snapshot values finite:",
      bool(np.isfinite(first_snapshot).all()))
print("All final-snapshot values finite:",
      bool(np.isfinite(final_snapshot).all()))

First snapshot: 0.0 (10792, 14)
Final snapshot: 1720.721256 (10792, 14)
All first-snapshot values finite: True
All final-snapshot values finite: True


In [36]:
restart_particles_ratio_1 = (
    restart_results["ratio_1"]["particles"]
    .sort_values("id")
    .reset_index(drop=True)
)

column_checks = []

restart_arrays = {
    "position": restart_particles_ratio_1[
        ["x", "y", "z"]
    ].to_numpy(),
    "velocity": restart_particles_ratio_1[
        ["vx", "vy", "vz"]
    ].to_numpy(),
    "radius": restart_particles_ratio_1[
        ["radius"]
    ].to_numpy(),
    "angular_velocity": restart_particles_ratio_1[
        ["omega_x", "omega_y", "omega_z"]
    ].to_numpy(),
}

data_arrays = {
    "position": final_snapshot[:, 0:3],
    "velocity": final_snapshot[:, 3:6],
    "radius": final_snapshot[:, 6:7],
    "angular_velocity": final_snapshot[:, 10:13],
}

for quantity in restart_arrays:
    difference = (
        data_arrays[quantity]
        - restart_arrays[quantity]
    )

    column_checks.append(
        {
            "quantity": quantity,
            "maximum_absolute_difference":
                np.max(np.abs(difference)),
            "root_mean_squared_difference":
                np.sqrt(np.mean(difference**2)),
            "all_close": np.allclose(
                data_arrays[quantity],
                restart_arrays[quantity],
                rtol=1e-8,
                atol=1e-10,
            ),
        }
    )

display(pd.DataFrame(column_checks))

restart_species = (
    restart_particles_ratio_1["species_index"]
    .to_numpy()
)

print(
    "Final data species agree with restart:",
    bool(np.array_equal(
        final_snapshot[:, 13].astype(int),
        restart_species,
    ))
)

print(
    "Radius sequence unchanged from first to final snapshot:",
    bool(np.array_equal(
        first_snapshot[:, 6],
        final_snapshot[:, 6],
    ))
)

print(
    "Species sequence unchanged from first to final snapshot:",
    bool(np.array_equal(
        first_snapshot[:, 13].astype(int),
        final_snapshot[:, 13].astype(int),
    ))
)

,quantity,maximum_absolute_difference,root_mean_squared_difference,all_close
0,position,0.000000e+00,0.000000e+00,True
1,velocity,4.999599e-10,1.008036e-10,True
2,radius,4.999401e-11,2.912491e-11,True
3,angular_velocity,4.997101e-10,4.833620e-11,True


Final data species agree with restart: True
Radius sequence unchanged from first to final snapshot: True
Species sequence unchanged from first to final snapshot: True


In [37]:
sample_snapshot_indices = np.unique(
    np.linspace(
        0,
        len(snapshot_index) - 1,
        11,
        dtype=int,
    )
)

ordering_checks = []

reference_radius = first_snapshot[:, 6]
reference_species = first_snapshot[:, 13].astype(int)

for snapshot_number in sample_snapshot_indices:
    row = snapshot_index.iloc[snapshot_number]

    time, snapshot = read_particle_snapshot(
        pilot_data_path,
        row["byte_offset"],
    )

    ordering_checks.append(
        {
            "snapshot_number": snapshot_number,
            "time": time,
            "particle_count": snapshot.shape[0],
            "all_values_finite":
                np.isfinite(snapshot).all(),
            "radius_sequence_identical":
                np.array_equal(
                    snapshot[:, 6],
                    reference_radius,
                ),
            "species_sequence_identical":
                np.array_equal(
                    snapshot[:, 13].astype(int),
                    reference_species,
                ),
            "minimum_x": snapshot[:, 0].min(),
            "maximum_x": snapshot[:, 0].max(),
            "minimum_y": snapshot[:, 1].min(),
            "maximum_y": snapshot[:, 1].max(),
            "minimum_z": snapshot[:, 2].min(),
            "maximum_z": snapshot[:, 2].max(),
        }
    )

ordering_validation = pd.DataFrame(ordering_checks)
display(ordering_validation)

print(
    "All sampled snapshots preserve radius ordering:",
    bool(
        ordering_validation[
            "radius_sequence_identical"
        ].all()
    )
)

print(
    "All sampled snapshots preserve species ordering:",
    bool(
        ordering_validation[
            "species_sequence_identical"
        ].all()
    )
)

,snapshot_number,time,particle_count,all_values_finite,radius_sequence_identical,species_sequence_identical,minimum_x,maximum_x,minimum_y,maximum_y,minimum_z,maximum_z
0,0,0.000000,10792,True,True,True,-28.368145,28.300724,-4.524970,4.524772,-29.521091,-6.104037
1,100,172.051606,10792,True,True,True,-29.516274,20.493477,-4.524792,4.524909,-29.514182,17.264452
2,200,344.103212,10792,True,True,True,-29.518539,21.703556,-4.524989,4.524882,-29.519114,18.791296
3,300,516.154818,10792,True,True,True,-29.519947,20.458772,-4.524846,4.525153,-29.512764,17.721494
4,400,688.206424,10792,True,True,True,-29.517981,21.144268,-4.524887,4.524929,-29.520028,17.889181
5,500,860.258030,10792,True,True,True,-29.509915,19.085449,-4.524828,4.525015,-29.513167,16.805640
6,600,1032.309636,10792,True,True,True,-29.522461,20.675989,-4.524837,4.524987,-29.523586,17.507348
7,700,1204.361241,10792,True,True,True,-29.514316,20.996293,-4.525010,4.524942,-29.522326,17.038341
8,800,1376.412847,10792,True,True,True,-29.520666,20.198461,-4.524817,4.525082,-29.518911,18.189149
9,900,1548.464453,10792,True,True,True,-29.522174,19.826753,-4.525000,4.524779,-29.518705,16.878912


All sampled snapshots preserve radius ordering: True
All sampled snapshots preserve species ordering: True


In [38]:
ROUTE_A_TIME_MIN = 1377.0

stationary_snapshot_index = snapshot_index.loc[
    snapshot_index["time"] >= ROUTE_A_TIME_MIN
].copy()

stationary_snapshot_index = (
    stationary_snapshot_index
    .reset_index(drop=True)
)

print(
    "Stationary snapshots:",
    len(stationary_snapshot_index)
)
print(
    "First retained time:",
    stationary_snapshot_index["time"].iloc[0]
)
print(
    "Final retained time:",
    stationary_snapshot_index["time"].iloc[-1]
)
print(
    "Stationary duration:",
    stationary_snapshot_index["time"].iloc[-1]
    - stationary_snapshot_index["time"].iloc[0]
)

print("\nRetained time-spacing summary:")
display(
    pd.Series(
        np.diff(
            stationary_snapshot_index[
                "time"
            ].to_numpy()
        )
    ).describe()
)

Stationary snapshots: 201
First retained time: 1378.133363
Final retained time: 1720.721256
Stationary duration: 342.5878930000001

Retained time-spacing summary:


count    200.000000
mean       1.712939
std        0.107149
min        0.205197
25%        1.720516
50%        1.720516
75%        1.720516
max        1.720517
dtype: float64

In [39]:
particle_count = int(
    snapshot_index["particle_count"].iloc[0]
)

representation_options = pd.DataFrame(
    [
        {
            "representation": "XZ position",
            "degrees_of_freedom": 2 * particle_count,
        },
        {
            "representation": "XYZ position",
            "degrees_of_freedom": 3 * particle_count,
        },
        {
            "representation": "XYZ position + velocity",
            "degrees_of_freedom": 6 * particle_count,
        },
    ]
)

representation_options[
    "stationary_float64_MiB"
] = (
    representation_options["degrees_of_freedom"]
    * len(stationary_snapshot_index)
    * 8
    / 1024**2
)

representation_options[
    "all_times_float64_MiB"
] = (
    representation_options["degrees_of_freedom"]
    * len(snapshot_index)
    * 8
    / 1024**2
)

display(representation_options)

,representation,degrees_of_freedom,stationary_float64_MiB,all_times_float64_MiB
0,XZ position,21584,33.099243,165.002197
1,XYZ position,32376,49.648865,247.503296
2,XYZ position + velocity,64752,99.297729,495.006592


In [40]:
import json

snapshot_index_path = (
    tables_directory
    / "particle_snapshot_index_ratio_1_000000_num_1.csv"
)

snapshot_index.to_csv(
    snapshot_index_path,
    index=False,
)

validation_summary = {
    "source_file": pilot_data_path.name,
    "source_size_bytes": pilot_data_path.stat().st_size,
    "snapshot_count": int(len(snapshot_index)),
    "particle_count": particle_count,
    "first_time": float(snapshot_index["time"].iloc[0]),
    "final_time": float(snapshot_index["time"].iloc[-1]),
    "timestamps_match_energy": bool(
        np.array_equal(
            snapshot_index["time"].to_numpy(),
            energy["time"].to_numpy(),
        )
    ),
    "sampled_ordering_checks": int(
        len(ordering_validation)
    ),
    "all_sampled_radius_sequences_identical": bool(
        ordering_validation[
            "radius_sequence_identical"
        ].all()
    ),
    "all_sampled_species_sequences_identical": bool(
        ordering_validation[
            "species_sequence_identical"
        ].all()
    ),
    "stationary_threshold": ROUTE_A_TIME_MIN,
    "stationary_snapshot_count": int(
        len(stationary_snapshot_index)
    ),
}

validation_path = (
    tables_directory
    / "particle_snapshot_validation_ratio_1_000000_num_1.json"
)

with validation_path.open("w") as handle:
    json.dump(
        validation_summary,
        handle,
        indent=2,
    )

print("Saved:", snapshot_index_path)
print("Saved:", validation_path)
print()
print(json.dumps(validation_summary, indent=2))

Saved: /Users/rallen/Documents/msc-rom-particle-systems/results/tables/particle_snapshot_index_ratio_1_000000_num_1.csv
Saved: /Users/rallen/Documents/msc-rom-particle-systems/results/tables/particle_snapshot_validation_ratio_1_000000_num_1.json

{
  "source_file": "rotating_drum_bidisperse_size_ratio_1_000000_num_1.data",
  "source_size_bytes": 2076613186,
  "snapshot_count": 1002,
  "particle_count": 10792,
  "first_time": 0.0,
  "final_time": 1720.721256,
  "timestamps_match_energy": true,
  "sampled_ordering_checks": 11,
  "all_sampled_radius_sequences_identical": true,
  "all_sampled_species_sequences_identical": true,
  "stationary_threshold": 1377.0,
  "stationary_snapshot_count": 201
}


## Conclusions and limitations

### Results established

- The raw training set contains 100 complete simulation bundles and the
  reserved test set contains 10 complete bundles.
- The complete cloud dataset has a logical size of approximately 282.56 GiB,
  of which the particle `.data` trajectories account for about 99%.
- Training size ratios span 1.0 to 2.0, while the ten reserved test ratios do
  not exactly overlap the training grid.
- Particle number varies across the parameter domain: the two inspected
  endpoint simulations contain 10,792 and 20,449 particles respectively.
  Consequently, an ordinary global particle-coordinate snapshot matrix cannot
  be constructed across all size ratios without an additional
  variable-cardinality or local-basis treatment.
- The size-ratio-1 pilot trajectory contains 1,002 snapshots and 10,792
  particles at every time.
- Snapshot times agree exactly with the corresponding MercuryDPM energy file.
- Particle radius and species sequences are unchanged across eleven sampled
  snapshots.
- The final particle record order agrees with restart-file particle-ID order.
  This provides strong evidence that row order supplies a consistent
  particle correspondence within this simulation.
- The interval used by the existing coarse-grained calculation, `time >= 1377`,
  contains 201 particle snapshots.

### File interpretation

Each particle snapshot begins with the particle count, time and fixed domain
bounds. Each particle row contains 14 values representing position, velocity,
radius, rotational/orientation information, angular velocity and species.
The precise interpretation of columns 7--9 remains to be confirmed, but those
columns are not required for the initial translational particle POD.

### Duplicate and auxiliary data

The additional
`rotating_drum_bidisperse_size_ratio_1_000000_w1.stat` file is byte-for-byte
identical to the corresponding `.1_w1.stat` file and should not be counted as
an independent coarse-graining experiment.

The `CG_windows` folder contains ten numbered temporal windows for each of ten
selected training simulations. `CG_windows_sigma` contains 127 distinct time
intervals for size ratio 1.303030, simulation 31. Its filename structure
demonstrates a temporal-window investigation, but the precise meaning of
`sigma` has not been assumed.

### Limitations

- Stable row ordering has been checked at eleven representative snapshots,
  rather than proven from explicit IDs stored in every `.data` record.
- Only the endpoint restart states and one full temporal trajectory have been
  inspected.
- Particle count varies between parameter values, preventing direct global
  parametric POD in raw particle coordinates.
- No POD, particle reconstruction, coarse-graining of reconstructed particles,
  or route comparison has yet been performed.
- The existing Route A model uses parameter-indexed, time-averaged continuum
  snapshots. A fair rank comparison with temporal particle POD will require a
  new controlled continuum POD using the same temporal snapshots.